# Colocalisation analysis of ER-GFP and ConA in cellular blebs

This notebook quantifies the colocalisation of two fluorescence channels inside
manually outlined regions of interest (ROIs, here: cellular blebs) and produces the
figures used in the manuscript of master thesis.

**Inputs** (placed in `data/`; every image needs a ROI archive with the same name)

```
data/
├── cell1.tif    # multichannel TIFF (ImageJ / OME-TIFF)
├── cell1.zip    # Fiji ROI Manager set (More > Save...) with the bleb ROIs
├── cell2.tif
├── cell2.zip
└── ...
```

**Outputs** (written to `results/`)

| File | Content |
|---|---|
| `colocalisation_per_bleb.csv` | Pearson, Spearman, Manders M1/M2 and Costes thresholds for every ROI |
| `colocalisation_summary.csv` | Mean, SEM and *n* per cell |
| `fluorograms_per_cell.png` | 2D intensity histogram per cell |
| `pseudocolour_maps.png` | Channel, merge and colocalisation maps for all cells |
| `fluorogram_all_cells.png` | Pooled scatter plot with one regression line per cell |
| `bar_graph.png` | Mean ± SEM per cell and for all blebs |
| `boxplot_all_blebs.png` | Box-and-whisker plot (min to max) of all blebs |
| `figure_colocalisation.png/.pdf` | Composite figure (panels A–C) |

**Sections:** 1 Setup · 2 Functions · 3 Analysis · 4 Figures 

> All display settings (percentile limits, gamma, colour maps) only affect the figures.
> Every coefficient is calculated from the raw, unscaled pixel intensities.

## 1. Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import roifile
import tifffile
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from scipy import stats
from skimage.draw import polygon

plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "pdf.fonttype": 42, "svg.fonttype": "none",   # editable text in PDF/SVG
})

In [ ]:
# Input / output
DATA_DIR = Path("data")       # folder with <name>.tif + <name>.zip
OUT_DIR = Path("results")     # all tables and figures are written here
OUT_DIR.mkdir(exist_ok=True)

# Channels 
CH_A, CH_B = 1, 2             # 0-based indices: channel 2 and channel 3
NAME_A, NAME_B = "ER", "ConA" # labels used in the figures

# Acquisition 
PIXEL_SIZE_UM = None          # None = read from the file metadata,
                              # otherwise set manually in um, e.g. 0.108

# Analysis
MIN_ROI_PIXELS = 10           # ROIs smaller than this are ignored

# Display only (never used for statistics)
DISPLAY_LO_PCT, DISPLAY_HI_PCT = 0.1, 99.9  # intensity limits, percentiles of the whole image
DISPLAY_GAMMA = 1.0           # < 1 brightens dim structures
MAP_CMAP = "turbo"            # colour map of the colocalisation map
MAP_VMAX_PCT = 99             # top of the colour scale = this percentile of A x B inside the ROIs
MAP_BG_DIM = 0.35             # brightness of the merged cell image behind the map (0 = black)
SCALEBAR_UM = 10              # scale bar length in um
SHOW_SCALEBAR_LABEL = False   # True: print the length next to the scale bar

SCATTER_NORM_PCT = 99.5       # fluorogram: divide intensities by this percentile of each cell
SCATTER_MAX_PTS = 15000       # fluorogram: plotted pixels per cell (statistics use all pixels)
BOX_YLIM = (0.5, 1.02)        # NOTE: truncated y axis in the box plot; state this in the legend
COMPOSITE_CELL = 0            # which cell is shown in the top row of the composite figure
RNG_SEED = 0                  # reproducible sub-sampling and jitter

## 2. Functions

In [ ]:
# Image and ROI I/O 
UNIT_TO_UM = {"nm": 1e-3, "um": 1.0, "\u00b5m": 1.0, "\u03bcm": 1.0, "micron": 1.0, "mm": 1e3}


def load_stack(path):
    """Read a TIFF and return a float array with axes (C, Z, Y, X)."""
    with tifffile.TiffFile(path) as tf:
        series = tf.series[0]
        arr, axes = series.asarray(), series.axes.replace("S", "C")
    if "C" not in axes and "Q" in axes:           # plain multipage TIFF: pages = channels
        axes = axes.replace("Q", "C", 1)
    for i in reversed(range(len(axes))):          # drop any other axis (e.g. T): keep first frame
        if axes[i] not in "ZCYX":
            arr = np.take(arr, 0, axis=i)
            axes = axes[:i] + axes[i + 1:]
    if "C" not in axes:
        raise ValueError(f"{Path(path).name}: no channel axis found (axes = {axes!r})")
    if "Z" not in axes:
        arr, axes = arr[np.newaxis], "Z" + axes
    arr = np.transpose(arr, [axes.index(a) for a in "CZYX"])
    return arr.astype(np.float64)


def get_pixel_size_um(path):
    """Pixel size in um from OME-XML or ImageJ metadata (None if not found)."""
    if PIXEL_SIZE_UM is not None:
        return PIXEL_SIZE_UM
    with tifffile.TiffFile(path) as tf:
        if tf.is_ome:                                         # e.g. Micro-Manager
            xml = tf.ome_metadata
            m = re.search(r'PhysicalSizeX="([\d.eE+-]+)"', xml)
            if m:
                u = re.search(r'PhysicalSizeXUnit="([^"]+)"', xml)
                factor = UNIT_TO_UM.get(u.group(1) if u else "\u00b5m")
                return float(m.group(1)) * factor if factor else None
        if tf.is_imagej:                                      # Fiji / ImageJ
            unit = (tf.imagej_metadata or {}).get("unit")
            tag = tf.pages[0].tags.get("XResolution")
            if tag and unit in UNIT_TO_UM:
                num, den = tag.value                          # pixels per unit
                if num:
                    return den / num * UNIT_TO_UM[unit]
    return None


def roi_mask(roi, shape):
    """Boolean mask (Y, X) of an ImageJ ROI."""
    xy = roi.coordinates()                                    # (N, 2) = x, y
    mask = np.zeros(shape, bool)
    rr, cc = polygon(xy[:, 1], xy[:, 0], shape)
    mask[rr, cc] = True
    return mask

In [ ]:
# Colocalisation metrics 
def costes_thresholds(a, b):
    """Automatic Costes thresholds (as in Fiji's Coloc 2).

    A least-squares line b = slope * a + intercept is fitted and the threshold is
    lowered stepwise along it until Pearson's r of all pixels below both thresholds
    is <= 0.
    """
    slope, intercept = np.polyfit(a, b, 1)
    for ta in np.linspace(a.max(), a.min(), 256):
        tb = slope * ta + intercept
        below = (a < ta) & (b < tb)
        if below.sum() < 3 or np.std(a[below]) == 0 or np.std(b[below]) == 0:
            continue
        if stats.pearsonr(a[below], b[below])[0] <= 0:
            return ta, tb
    return 0.0, 0.0


def coloc_metrics(a, b):
    """Pearson, Spearman and Manders (Costes-thresholded) for two 1D pixel arrays."""
    ta, tb = costes_thresholds(a, b)
    return {
        "pearson": stats.pearsonr(a, b)[0],
        "spearman": stats.spearmanr(a, b)[0],
        "M1": a[b > tb].sum() / a.sum(),    # fraction of channel A overlapping channel B
        "M2": b[a > ta].sum() / b.sum(),    # fraction of channel B overlapping channel A
        "thr_A": ta,
        "thr_B": tb,
    }

In [ ]:
# Plot helpers
def display_scale(img):
    """Scale an image to 0-1 for display using percentiles of the WHOLE image."""
    vmin, vmax = np.percentile(img, [DISPLAY_LO_PCT, DISPLAY_HI_PCT])
    return np.clip((img - vmin) / (vmax - vmin + 1e-12), 0, 1) ** DISPLAY_GAMMA


def crop_box(mask, pad=20):
    """Slices of the bounding box of a mask plus a margin."""
    ys, xs = np.where(mask)
    return (slice(max(ys.min() - pad, 0), ys.max() + pad),
            slice(max(xs.min() - pad, 0), xs.max() + pad))


def add_scalebar(ax, px_um, length_um, shape, color="white"):
    """Draw a scale bar (bottom right). Does nothing if the pixel size is unknown."""
    if not px_um:
        return
    h, w = shape
    length_px = length_um / px_um
    x0, y0 = w - length_px - 0.05 * w, h - 0.06 * h
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    ax.plot([x0, x0 + length_px], [y0, y0], color=color, lw=3, solid_capstyle="butt")
    if SHOW_SCALEBAR_LABEL:
        ax.text(x0 + length_px / 2, y0 - 0.02 * h, f"{length_um:g} \u00b5m",
                color=color, ha="center", va="bottom", fontsize=9)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


def panel_letter(ax, letter, x=-0.18, y=1.06):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=14, fontweight="bold")


def save_fig(fig, name, formats=("png",)):
    for fmt in formats:
        fig.savefig(OUT_DIR / f"{name}.{fmt}", dpi=300, bbox_inches="tight")


def draw_cell_panels(fig, axes, cell, titles):
    """Four panels for one cell: channel A, channel B, merge and colocalisation map."""
    sl = crop_box(cell["union"])
    A = display_scale(cell["img_a"])[sl]
    B = display_scale(cell["img_b"])[sl]
    mask = cell["union"][sl]
    prod = A * B
    vmax = max(np.percentile(prod[mask], MAP_VMAX_PCT), 1e-6)
    zeros = np.zeros_like(A)

    images = [np.dstack([zeros, A, zeros]),        # channel A: green
              np.dstack([B, zeros, B]),            # channel B: magenta
              np.dstack([B, A, B])]                # merge
    for ax, img, title in zip(axes[:3], images, titles[:3]):
        ax.imshow(img)
        ax.set_title(title)
        add_scalebar(ax, cell["px_um"], SCALEBAR_UM, A.shape, "white")

    ax = axes[3]                                   # colocalisation map on a dimmed merge
    ax.imshow(images[2] * MAP_BG_DIM)
    im = ax.imshow(np.ma.masked_where(~mask, prod), cmap=MAP_CMAP, vmin=0, vmax=vmax)
    ax.set_title(titles[3])
    add_scalebar(ax, cell["px_um"], SCALEBAR_UM, A.shape, "white")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=f"{NAME_A} \u00d7 {NAME_B} (a.u.)")
    for ax in axes:
        ax.axis("off")


def draw_pooled_scatter(ax, rng):
    """Fluorogram of all cells: points are shuffled, one regression line per cell."""
    xs, ys, colours, handles = [], [], [], []
    for i, name in enumerate(names):
        a = cells[name]["pix_a"] / np.percentile(cells[name]["pix_a"], SCATTER_NORM_PCT)
        b = cells[name]["pix_b"] / np.percentile(cells[name]["pix_b"], SCATTER_NORM_PCT)
        r = stats.pearsonr(a, b)[0]
        idx = rng.choice(a.size, min(SCATTER_MAX_PTS, a.size), replace=False)
        xs.append(a[idx]); ys.append(b[idx]); colours += [f"C{i}"] * idx.size
        slope, intercept = np.polyfit(a, b, 1)
        line_x = np.array([a.min(), a.max()])
        ax.plot(line_x, slope * line_x + intercept, color=f"C{i}", lw=2, zorder=4)
        handles.append(Line2D([], [], marker="o", ls="", color=f"C{i}",
                              label=f"{labels[name]} (r = {r:.2f})"))
    xs, ys, colours = np.concatenate(xs), np.concatenate(ys), np.array(colours)
    order = rng.permutation(xs.size)               # shuffle so no cell hides the others
    ax.scatter(xs[order], ys[order], s=2, c=colours[order], alpha=0.25,
               linewidths=0, rasterized=True)
    ax.set_xlim(0, 1.3); ax.set_ylim(0, 1.3)
    ax.set_box_aspect(1)
    ax.set_xlabel(f"{NAME_A} normalised intensity")
    ax.set_ylabel(f"{NAME_B} normalised intensity")
    ax.legend(handles=handles, frameon=False, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)


def draw_boxplot(ax, rng):
    """Box (quartiles, median) with whiskers from minimum to maximum and all blebs as dots."""
    data = [df[key].dropna().values for key, _ in METRICS]
    bp = ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=False, whis=(0, 100),
                    medianprops=dict(color="black", lw=1.8),
                    whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2))
    for patch, colour in zip(bp["boxes"], ["C0", "C1", "C2"]):
        patch.set(facecolor=colour, alpha=0.6, edgecolor="black")
    for i, vals in enumerate(data, start=1):
        ax.scatter(i + rng.uniform(-0.12, 0.12, vals.size), vals,
                   s=12, color="black", alpha=0.6, zorder=3)
    ax.set_xticks(range(1, len(METRICS) + 1))
    ax.set_xticklabels([label.replace(" (", "\n(") for _, label in METRICS])
    ax.set_ylim(*BOX_YLIM)
    ax.set_ylabel("Coefficient value")
    ax.set_box_aspect(1)
    ax.spines[["top", "right"]].set_visible(False)

## 3. Analysis

For every `<name>.tif` the ROIs in `<name>.zip` are applied to channels `CH_A` and `CH_B`.
ROIs assigned to a z-plane in Fiji are analysed on that plane; all other ROIs, and
single-plane images, use the maximum-intensity projection. ROIs with fewer than
`MIN_ROI_PIXELS` pixels or without intensity variance in either channel are skipped.

In [ ]:
cells = {}   # per-cell data used for the figures
rows = []    # one row per bleb

tif_files = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in {".tif", ".tiff"})
if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {DATA_DIR.resolve()}")

for tif in tif_files:
    roi_path = tif.with_suffix(".zip")
    if not roi_path.exists():
        print(f"[skipped] {tif.name}: no ROI archive named {roi_path.name}")
        continue

    stack = load_stack(tif)                                   # (C, Z, Y, X)
    n_ch, n_z, ny, nx = stack.shape
    if max(CH_A, CH_B) >= n_ch:
        raise ValueError(f"{tif.name}: has {n_ch} channels, but CH_A/CH_B = {CH_A}/{CH_B}")

    rois = roifile.roiread(roi_path)
    rois = rois if isinstance(rois, list) else [rois]

    pix_a, pix_b, masks = [], [], []
    for i, roi in enumerate(rois, start=1):
        z = getattr(roi, "z_position", 0) or roi.position
        if n_z > 1 and 1 <= z <= n_z:                         # ROI assigned to a z-plane
            plane_a, plane_b = stack[CH_A, z - 1], stack[CH_B, z - 1]
        else:                                                 # otherwise: max projection
            plane_a, plane_b = stack[CH_A].max(0), stack[CH_B].max(0)

        mask = roi_mask(roi, (ny, nx))
        if mask.sum() < MIN_ROI_PIXELS:
            continue
        a, b = plane_a[mask], plane_b[mask]
        if np.std(a) == 0 or np.std(b) == 0:
            continue

        rows.append({"cell": tif.stem, "bleb": roi.name or f"roi{i}",
                     "n_pixels": int(mask.sum()), **coloc_metrics(a, b)})
        pix_a.append(a); pix_b.append(b); masks.append(mask)

    if not masks:
        print(f"[skipped] {tif.name}: no valid ROIs")
        continue

    cells[tif.stem] = {
        "img_a": stack[CH_A].max(0),                          # for display only
        "img_b": stack[CH_B].max(0),
        "union": np.any(masks, axis=0),
        "pix_a": np.concatenate(pix_a),                       # all ROI pixels of this cell
        "pix_b": np.concatenate(pix_b),
        "px_um": get_pixel_size_um(tif),
    }
    px = cells[tif.stem]["px_um"]
    print(f"{tif.stem}: {len(masks)} blebs, pixel size = {f'{px:.4g} um' if px else 'unknown'}")

if not cells:
    raise RuntimeError("No cells were analysed; check file names and ROI archives.")

names = list(cells)
labels = {name: f"Cell {i}" for i, name in enumerate(names, start=1)}

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "colocalisation_per_bleb.csv", index=False)

COEFFS = ["pearson", "spearman", "M1", "M2"]
summary = df.groupby("cell")[COEFFS].agg(["mean", "sem", "count"])
summary.to_csv(OUT_DIR / "colocalisation_summary.csv")

METRICS = [("pearson", "Pearson's r"),
           ("M1", f"Manders M1 ({NAME_A})"),
           ("M2", f"Manders M2 ({NAME_B})")]

print(f"\nTotal: {len(df)} blebs in {len(names)} cells")
summary.round(3)

## 4. Figures

### 4.1 Fluorogram per cell (2D intensity histogram of all bleb pixels)

In [ ]:
fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 4.6), squeeze=False)
for ax, name in zip(axes[0], names):
    a, b = cells[name]["pix_a"], cells[name]["pix_b"]
    hist = ax.hist2d(a, b, bins=128, cmap="viridis", norm=LogNorm())
    slope, intercept = np.polyfit(a, b, 1)
    line_x = np.array([a.min(), a.max()])
    ax.plot(line_x, slope * line_x + intercept, "r--", lw=1.5)
    ax.set_title(f"{labels[name]}\nPearson r = {stats.pearsonr(a, b)[0]:.3f}")
    ax.set_xlabel(f"{NAME_A} intensity")
    ax.set_ylabel(f"{NAME_B} intensity")
    fig.colorbar(hist[3], ax=ax, label="pixel count")
fig.tight_layout()
save_fig(fig, "fluorograms_per_cell")
plt.show()

### 4.2 Pseudo-colour maps (all cells)

In [ ]:
fig, axes = plt.subplots(len(names), 4, figsize=(16, 4 * len(names)), squeeze=False)
for row, name in zip(axes, names):
    draw_cell_panels(fig, row, cells[name], [
        f"{labels[name]} \u2013 {NAME_A}", f"{labels[name]} \u2013 {NAME_B}", "Merge",
        f"Colocalisation map ({NAME_A} \u00d7 {NAME_B})"])
fig.tight_layout()
save_fig(fig, "pseudocolour_maps")
plt.show()

### 4.3 Fluorogram of all cells

In [ ]:
rng = np.random.default_rng(RNG_SEED)
fig, ax = plt.subplots(figsize=(6.5, 6))
draw_pooled_scatter(ax, rng)
fig.tight_layout()
save_fig(fig, "fluorogram_all_cells")
plt.show()

### 4.4 Bar graph (mean ± SEM)

In [ ]:
rng = np.random.default_rng(RNG_SEED)
groups = names + ["All blebs"]
width = 0.8 / len(groups)

fig, ax = plt.subplots(figsize=(9, 5))
for gi, group in enumerate(groups):
    is_all = group == "All blebs"
    for mi, (key, _) in enumerate(METRICS):
        vals = (df[key] if is_all else df.loc[df.cell == group, key]).dropna().values
        x = mi + (gi - (len(groups) - 1) / 2) * width
        sem = stats.sem(vals) if vals.size > 1 else 0
        ax.bar(x, vals.mean(), width * 0.9, yerr=sem, capsize=3, edgecolor="black", alpha=0.85,
               color="lightgray" if is_all else f"C{gi}",
               label=("All blebs" if is_all else labels[group]) if mi == 0 else None)
        ax.scatter(x + rng.uniform(-width / 4, width / 4, vals.size), vals,
                   s=8, color="black", zorder=3)
ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels([label for _, label in METRICS])
ax.set_ylabel("Coefficient value")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_fig(fig, "bar_graph")
plt.show()

### 4.5 Box plot (all blebs, minimum to maximum)

In [ ]:
rng = np.random.default_rng(RNG_SEED)
fig, ax = plt.subplots(figsize=(6, 5))
draw_boxplot(ax, rng)
fig.tight_layout()
save_fig(fig, "boxplot_all_blebs")
plt.show()

### 4.6 Composite figure

Panel A: one representative cell (`COMPOSITE_CELL`); panel B: fluorogram of all cells;
panel C: box plot of all blebs.

In [ ]:
rng = np.random.default_rng(RNG_SEED)

fig = plt.figure(figsize=(11, 9.5))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.15], hspace=0.30, wspace=0.12,
                      left=0.08, right=0.97, top=0.95, bottom=0.08)

# A: representative cell
axes_top = [fig.add_subplot(gs[0, k]) for k in range(4)]
draw_cell_panels(fig, axes_top, cells[names[COMPOSITE_CELL]],
                 [NAME_A, NAME_B, "Merge", f"Colocalisation map ({NAME_A} \u00d7 {NAME_B})"])
panel_letter(axes_top[0], "A", x=-0.05, y=1.08)

# B: fluorogram, all cells
ax_scatter = fig.add_subplot(gs[1, 0:2])
draw_pooled_scatter(ax_scatter, rng)
panel_letter(ax_scatter, "B")

# C: box plot, all blebs
ax_box = fig.add_subplot(gs[1, 2:4])
draw_boxplot(ax_box, rng)
panel_letter(ax_box, "C")

save_fig(fig, "figure_colocalisation", formats=("png", "pdf"))
plt.show()